# 🔐 CyberNews Scraper — Google Colab

Pipeline para recopilar, puntuar y distribuir las **4 noticias más relevantes de ciberseguridad** del mes.

### Fuentes
The Hacker News · SecurityWeek · Kaspersky Securelist · WeLiveSecurity ES

### Pasos
1. **Paso 1** – Cargar el código desde GitHub
2. **Paso 2** – Instalar dependencias
3. **Paso 3** – Ajustar parámetros de ejecución
4. **Paso 4** – Ejecutar el pipeline
5. **Paso 5** – Ver resultados en el notebook
6. **Paso 6** – Descargar archivos _(opcional)_

> El envío por email / Teams / Slack se configura en una fase posterior.


In [ ]:
#@title 🔧 Paso 1 – Clonar el proyecto desde GitHub

import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path('/content/cybernews-scraper')

# Volver a un directorio seguro ANTES de borrar (evita "No such file or directory")
os.chdir('/content')

if PROJECT_DIR.exists():
    shutil.rmtree(str(PROJECT_DIR))

result = subprocess.run(
    ['git', 'clone', 'https://github.com/dalesos92/cybernews_scraper.git', str(PROJECT_DIR)],
    capture_output=True,
    text=True,
)
print(result.stderr)  # git clone escribe progreso en stderr
if result.returncode != 0:
    raise RuntimeError('Clonado fallido. Revisa el mensaje de arriba.')

os.chdir(str(PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f'✓ Directorio: {os.getcwd()}')
print(f'  Archivos: {sorted(p.name for p in PROJECT_DIR.iterdir() if not p.name.startswith("."))[:12]}')


In [ ]:
#@title 📦 Paso 2 – Instalar dependencias

import subprocess
import os
from pathlib import Path

# Localizar requirements.txt: primero en el cwd establecido por el Paso 1,
# luego en la ruta fija de extracción, y finalmente en el directorio del notebook.
def _find_req():
    candidates = [
        Path(os.getcwd()) / 'requirements.txt',
        Path('/content/cybernews-scraper/requirements.txt'),
    ]
    try:
        candidates.insert(0, Path(PROJECT_DIR) / 'requirements.txt')
    except NameError:
        pass
    for p in candidates:
        if p.exists():
            return p
    return None

_req = _find_req()
if _req is None:
    raise FileNotFoundError(
        'No se encontró requirements.txt. '
        'Asegúrate de haber ejecutado el Paso 1 antes.'
    )

print(f'✓ requirements.txt encontrado en: {_req}')
print('Instalando dependencias...')
result = subprocess.run(
    ['pip', 'install', '-q', '-r', str(_req)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print('⚠ Advertencias / errores durante la instalación:')
    print((result.stderr or result.stdout)[-2000:])
else:
    print('✓ Dependencias instaladas correctamente')


In [ ]:
#@title ⚙️ Paso 3 – Parámetros de ejecución

#@markdown #### Contenido
top_n = 4  #@param {type:"integer"}
lookback_days = 30  #@param {type:"integer"}

#@markdown #### Fuentes activas (solo en español)
enable_welivesecurity = False   #@param {type:"boolean"}
enable_cybersecnews_es = True  #@param {type:"boolean"}
enable_hispasec = True  #@param {type:"boolean"}
enable_revista_ciberseguridad = True  #@param {type:"boolean"}
enable_incibe = False  #@param {type:"boolean"}
enable_seguinfo = True  #@param {type:"boolean"}
enable_kaspersky_latam = True  #@param {type:"boolean"}

print(f'✓ top_n={top_n} | lookback_days={lookback_days}')
_fuentes = [
    ('WeLiveSec',       enable_welivesecurity),
    ('CyberSecNewsES',  enable_cybersecnews_es),
    ('Hispasec',        enable_hispasec),
    ('RevistaCiber',    enable_revista_ciberseguridad),
    ('INCIBE',          enable_incibe),
    ('Segu-Info',       enable_seguinfo),
    ('KasperskyLATAM',  enable_kaspersky_latam),
]

print('  Fuentes: ' + ', '.join(f'{n}={"ON" if v else "off"}' for n, v in _fuentes))

In [ ]:
#@title 🚀 Paso 4 – Ejecutar pipeline

import os
from pathlib import Path

# Generar .env solo con las variables necesarias para ver noticias
_env_vars = {
    'LOG_LEVEL':                     'INFO',
    'OUTPUT_DIR':                    'output',
    'DB_PATH':                       'data/cybernews.db',
    'HTTP_TIMEOUT':                  '30',
    'HTTP_MAX_RETRIES':              '3',
    'TOP_N':                         str(top_n),
    'LOOKBACK_DAYS':                 str(lookback_days),
    'ENABLE_WELIVESECURITY':         str(enable_welivesecurity).lower(),
    'ENABLE_CYBERSECNEWS_ES':        str(enable_cybersecnews_es).lower(),
    'ENABLE_HISPASEC':               str(enable_hispasec).lower(),
    'ENABLE_REVISTA_CIBERSEGURIDAD': str(enable_revista_ciberseguridad).lower(),
    'ENABLE_INCIBE':                 str(enable_incibe).lower(),
    'ENABLE_SEGUINFO':               str(enable_seguinfo).lower(),
    'ENABLE_KASPERSKY_LATAM':        str(enable_kaspersky_latam).lower(),
}
Path('.env').write_text('\n'.join(f'{k}={v}' for k, v in _env_vars.items()), encoding='utf-8')
print('✓ .env generado')
print('=' * 60)

# --dry-run: genera archivos de salida pero no guarda en BD ni envía nada
os.system('python main.py --dry-run')


In [ ]:
#@title 📊 Paso 5 – Ver resultados

import json
from pathlib import Path
from IPython.display import HTML, Markdown, display

_out = Path('output')

# Resumen en texto
_json_path = _out / 'top4_monthly.json'
if _json_path.exists():
    _data = json.loads(_json_path.read_text(encoding='utf-8'))
    print(f"Generado : {_data['generated_at']}")
    print(f"Asunto   : {_data['subject']}")
    print()
    for item in _data['items']:
        print(f"  #{item['rank']} [{item['source']}]")
        print(f"     {item['title']}")
        print(f"     {item['url']}")
        print()
else:
    print('⚠ top4_monthly.json no encontrado. ¿Ejecutaste el Paso 4?')

# Markdown renderizado
_md_path = _out / 'top4_monthly.md'
if _md_path.exists():
    print('\n' + '─' * 50)
    display(Markdown(_md_path.read_text(encoding='utf-8')))

# Preview HTML del email
_html_path = _out / 'top4_email.html'
if _html_path.exists():
    print('\n' + '─' * 50)
    display(HTML(_html_path.read_text(encoding='utf-8')))


In [ ]:
#@title 💾 Paso 6 – Descargar archivos de salida (opcional)

from pathlib import Path

try:
    from google.colab import files as colab_files
    _found = 0
    for _fname in ['top4_monthly.json', 'top4_monthly.md', 'top4_email.html']:
        _fpath = Path('output') / _fname
        if _fpath.exists():
            colab_files.download(str(_fpath))
            print(f'✓ Descargando {_fname}')
            _found += 1
    if _found == 0:
        print('⚠ No se encontraron archivos. Ejecuta el Paso 4 primero.')
except ImportError:
    print('ℹ Esta celda solo funciona en Google Colab.')


In [ ]:
#@title ☁️ Paso 7 – Subir a Drive y enviar correo por Apps Script

#@markdown Requiere tener configurados en **Colab Secrets** (🔑):
#@markdown - `GOOGLE_DRIVE_FOLDER_ID` — ID de la carpeta Google Drive
#@markdown - `GOOGLE_APPSCRIPT_TOKEN` — token generado con `setupWebhookToken()`

# ── URL del Web App de Apps Script (no sensible — puede ir en el código) ──
APPSCRIPT_WEBHOOK_URL = 'REEMPLAZAR_CON_URL_DEL_WEB_APP'  # https://script.google.com/macros/s/.../exec
EMAIL_TEMPLATE        = 'c'  # Plantilla a subir: a | b | c

import os
from pathlib import Path

# ── Leer secretos desde Colab Secrets ──
try:
    from google.colab import userdata
    DRIVE_FOLDER_ID = userdata.get('GOOGLE_DRIVE_FOLDER_ID')
    APPSCRIPT_TOKEN = userdata.get('GOOGLE_APPSCRIPT_TOKEN')
except Exception as e:
    raise RuntimeError(
        'No se pudieron leer los secretos de Colab. '
        'Agrega GOOGLE_DRIVE_FOLDER_ID y GOOGLE_APPSCRIPT_TOKEN en el panel 🔑.'
    ) from e

if not DRIVE_FOLDER_ID:
    raise ValueError('GOOGLE_DRIVE_FOLDER_ID está vacío en Colab Secrets.')

# ── Autenticar con la cuenta Google del usuario de Colab ──
# Esto evita que google.auth.default() intente usar Compute Engine (GCE)
# y luego falle con TransportError / 404 en el metadata service.
from google.colab import auth as _colab_auth
_colab_auth.authenticate_user()

import google.auth
_DRIVE_SCOPES = ['https://www.googleapis.com/auth/drive']
_creds, _ = google.auth.default(scopes=_DRIVE_SCOPES)
print('✓ Autenticación Google completada')

# ── Determinar qué HTML subir según el template elegido ──
_html_map = {'a': 'top4_email_bbva_a.html', 'b': 'top4_email_bbva_b.html', 'c': 'top4_email_bbva_c.html'}
_html_name = _html_map.get(EMAIL_TEMPLATE, 'top4_email.html')
_html_path = Path('output') / _html_name
if not _html_path.exists():
    _html_path = Path('output/top4_email.html')  # fallback al genérico

_json_path = Path('output/top4_monthly.json')
_paths = [p for p in [_html_path, _json_path] if p.exists()]

if not _paths:
    raise FileNotFoundError('No se encontraron archivos de salida. Ejecuta el Paso 4 primero.')

# ── Subir a Google Drive ──
from src.drive_uploader import upload_artifacts

print('Subiendo archivos a Google Drive...')
ids = upload_artifacts(paths=_paths, folder_id=DRIVE_FOLDER_ID, credentials=_creds)
for name, fid in ids.items():
    print(f'  ✓ {name}  →  https://drive.google.com/file/d/{fid}/view')

# ── Disparar Apps Script para enviar el correo ──
if APPSCRIPT_WEBHOOK_URL.startswith('https://'):
    import httpx
    print('\nLlamando al Web App de Apps Script...')
    resp = httpx.post(
        APPSCRIPT_WEBHOOK_URL,
        json={'token': APPSCRIPT_TOKEN or ''},
        timeout=30,
        follow_redirects=True,
    )
    try:
        data = resp.json()
        if data.get('status') == 200:
            print(f"  ✓ {data['message']}")
        else:
            print(f"  ⚠ Apps Script error {data.get('status')}: {data.get('message')}")
    except Exception:
        print(f'  Respuesta inesperada ({resp.status_code}): {resp.text[:200]}')
else:
    print('\n⚠ APPSCRIPT_WEBHOOK_URL no configurada. Archivos subidos a Drive, pero el correo no se envió.')
    print('   Puedes dispararlo desde el menú del Sheet: 🔒 CyberNews → Enviar correo del mes')
